# Introduction to Physics Neural Surrogates for Engineering and Healthcare

### Student notebook · fill-in exercises

**ML in PL Conference 2026, 11 October 2026, 09:00–13:00**  

**Patryk Rygiel (p.t.rygiel@utwente.nl) · Julian Suk (julian.suk@tum.de)**

**Description:**   
Computational modelling of physical phenomena - such as fluid flow, structural mechanics, and weather systems - plays a central role in science, engineering, and healthcare. These phenomena are commonly modelled using high-fidelity numerical solvers that approximate the governing physical laws expressed through partial differential equations (PDEs). Although such methods provide accurate and physically grounded solutions, their computational cost and runtime remain major bottlenecks in many real-world applications.

In recent years, neural surrogates - deep learning-based approximators of numerical methods - have emerged as a promising alternative, offering orders-of-magnitude acceleration by amortising simulation cost into offline training. Once trained, they can estimate physical quantities of interest in seconds, enabling design, analysis, and discovery workflows at scale.

This tutorial will introduce the theoretical foundations and practical design choices involved in building physics-based neural surrogates for engineering and healthcare applications. We will begin with geometric deep learning and examine how neural networks can process large, unstructured 3D geometries commonly encountered in engineering and biomedical workflows. We will then introduce neural fields and neural operators, which provide key building blocks for scalable and discretisation-agnostic surrogate models. We will showcase how the neural operator properties can be achieved by recombining core components of the Transformer architecture. Finally, we will briefly discuss physical inductive biases, including conservation laws and operator symmetries, and explain how they can improve model reliability and data efficiency, particularly in data-scarce settings.

Each part of the tutorial will be accompanied by hands-on exercises based on relevant engineering and healthcare examples, allowing participants to connect the theoretical concepts with practical implementation.

**Goals of the tutorial:**   
This tutorial aims to give participants a practical introduction to physics-based neural surrogates for engineering and healthcare. By the end of the tutorial, attendees will be able to: understand the role of neural surrogates in physics modelling; work with complex 3D geometries and unstructured data; explain the principles of neural fields and neural operators and how to implement them using Transformer architecture; recognise how physical constraints and symmetries can improve reliability and data efficiency; and implement and evaluate basic neural surrogate models in PyTorch. Participants will leave with both a conceptual overview and practical experience that they can apply to their own research and engineering problems.

**Prerequisites:**   
Participants should have a basic understanding of deep learning, Transformer architecture, and PyTorch, as the examples and practical exercises will build on these concepts. The exercises will be delivered through Jupyter Notebooks, so participants should have access to either Google Colab or a suitable local Python environment. Prior knowledge of neural operators, partial differential equations, numerical simulation, or geometry processing is helpful but not required — the essential concepts will be introduced during the tutorial.

# Physics neural surrogates · cylinder tutorial
Student edition: implement the numbered TODOs before running dependent cells.

1. Mesh/radius GNNs and remeshing.
2. Coordinate neural fields.
3. Temporal operators and rollouts.
4. Cross-attention and anchor-only training.
5. Fixed-anchor query independence, changed conditioning, and full-mesh self-attention.

Keep cylinder_tutorial_utils.py beside this notebook. CFD/predicted speed limits are 0–8. Drift is measured between predictions, while CFD error is measured against the next ground-truth frame.


### Exercise map

| Exercise | Implement | Check |
|---|---|---|
| 1.2 | PyG batch and disjoint-edge assertion | No edges between examples |
| 1.3 | GNN updates and snapshot loss | Mesh/radius predictions |
| 2 | Field forward and snapshot loss | Remeshing invariance |
| 3a | Residual update and temporal loss | Next-frame prediction |
| 3b | Reseeded and pure rollout updates | Identical targets in plot/movie |
| 4a | Cross-attention projections and attention | Q from queries, K/V from context |
| 4b | Two residual attention/MLP stages | Queries never become anchor K/V |
| 4c | Uniform anchor-only loss | Only sampled anchors enter training |
| 5a | Query splitting and invariance check | Fixed anchors, unchanged retained queries |
| 5b | Replacement anchor set | Conditioning change can cause drift |
| 5c | Full-self forward and full-mesh loss | Every mesh vertex enters training |
| 5d | Restrict decoding points | Full-self predictions can change |

Data, plotting, remeshing and measured epoch profiling are supplied.


In [ ]:
import importlib.util, subprocess, sys

# requirements={"torch":"torch>=2.5,<3","torch_geometric":"torch-geometric==2.6.1","numpy":"numpy","scipy":"scipy","matplotlib":"matplotlib","pyvista":"pyvista","tqdm":"tqdm"}
# missing=[package for module,package in requirements.items() if importlib.util.find_spec(module) is None]
# if missing: subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])

from pathlib import Path
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv
import cylinder_tutorial_utils as U

importlib.reload(U)

SEED = 20261011
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
U.set_seed(SEED)


## 1. PyG and working with unstructured data
Use one extracted [CFDBench raw cylinder case](https://huggingface.co/datasets/chen-yingfa/CFDBench-raw/tree/main/04_cylinderflow). Set the path, cylinder geometry and frame. Raw exports have coordinates and fields but no CFD faces: the loader constructs a teaching triangulation and masks the cylinder hole. Frame indices are not physical seconds.

In [ ]:
CYLINDER_DATA = Path("../case40")
CYLINDER = (0.0, 0.0, 0.01)
CYLINDER_START = 200
CYLINDER_FRAMES = 40


### 1.1. Data object
Data object in PyG is a container for storing graph data, including node features, edge indices, and labels. In this case, we will create a `Dataset` that yields `Data` objects that contains:
- `data.pos`: the coordinates of the mesh vertices
- `data.x`: the node features (could be previous velocity, pressure, or any other relevant physical quantity)
- `data.y`: the corresponding velocity fields (labels)
- `data.edge_index`: the connectivity of the mesh (edges between vertices)
- `data.edge_attr`: optional edge attributes (e.g., distances or weights)
- `data.face`: the faces of the mesh (triangles connecting vertices)

In [ ]:
dataset = U.CylinderDataset(CYLINDER_DATA, cylinder=CYLINDER)
sample = dataset[CYLINDER_START]
print(sample)

_ = U.plot_data(sample)


In [ ]:
movie = U.animate_cylinder(
    dataset, start_step=CYLINDER_START, num_steps=CYLINDER_FRAMES, fps=30, field="speed"
)


### 1.2. Batching unstructured data

In [ ]:
# TODO 1.2: batch the two consecutive graphs.
raise NotImplementedError("Complete exercise 1.2")

print(batch)
print(batch.batch)

# Every edge must stay within one graph of the disjoint batch.
# TODO 1.2: assert edges stay within each graph.
raise NotImplementedError("Complete exercise 1.2")


### 1.3. Exercise 1: training GNNs

Train two identical architectures on mesh and pointcloud data structures. Explore the effect of connectivity on training and generalisation through remeshing tests.

In [ ]:
class FourierCoordinates(nn.Module):
    """Fourier-based embedding for postional encoding"""

    def __init__(self, dim, levels=3):
        super().__init__()
        self.register_buffer("freq", torch.pi * 2.0 ** torch.arange(levels))
        self.out_dim = dim * (1 + 2 * levels)

    def forward(self, pos):
        a = pos.unsqueeze(-1) * self.freq
        return torch.cat([pos, a.sin().flatten(-2), a.cos().flatten(-2)], dim=-1)


class SnapshotGNN(nn.Module):
    def __init__(self, width: int = 128):
        super().__init__()
        self.fourier = FourierCoordinates(2)  # Best practice
        self.conv1 = SAGEConv(14, width)
        self.conv2 = SAGEConv(width, width)
        self.head = nn.Sequential(nn.Linear(width, width), nn.SiLU(), nn.Linear(width, 2))

    def forward(self, data):
        # TODO 1.3: two GraphSAGE updates and output head.
        raise NotImplementedError("Complete exercise 1.3")


In [ ]:
POINT_RADIUS = 0.0002

graphs = {
    "mesh": Data(
        pos=sample.pos.clone(),
        x=sample.x.clone(),
        y=sample.y.clone(),
        face=sample.face.clone(),
        edge_index=U.face_edges(sample.face, sample.num_nodes),
    ),
    "pointcloud": Data(
        pos=sample.pos.clone(),
        x=sample.x.clone(),
        y=sample.y.clone(),
        face=sample.face.clone(),
        edge_index=U.radius_edges(sample.pos, POINT_RADIUS),
    ),
}
for name, g in graphs.items():
    print(name, g.num_nodes, "nodes", g.num_edges, "directed edges")

_ = U.plot_graph_connectivities(graphs, POINT_RADIUS, node_id=6000)


In [ ]:
def snapshot_loss(model):
    # TODO 1.3: next-snapshot MSE against g.y.
    raise NotImplementedError("Complete exercise 1.3")


TRAIN_STEPS = 2000
models, histories = {}, {}

for name, graph in graphs.items():
    U.set_seed(SEED + 1)
    model = SnapshotGNN().to(DEVICE)
    g = U.on_device(graph, DEVICE)

    histories[name] = U.train_loop(model, snapshot_loss, TRAIN_STEPS, lr=1e-3)
    models[name] = model

original_predictions = {
    name: U.predict_graph(m, graphs[name], DEVICE) for name, m in models.items()
}

U.plot_history(histories)
_ = U.plot_predictions(sample, original_predictions)


### Two independent remeshing tests
**Larger elements:** merge adjacent triangles into polygons by deleting internal edges, keeping vertices and field values. Radius neighborhoods are identical.

**Edge collapse:** contract randomly selected eligible edges into an existing endpoint, reducing vertices. Rebuild both graphs at the same physical radius. Each successful collapse removes one vertex; the percentage refers to initial undirected edges. Boundary/topology checks may limit the achieved number.

Choose the parameters below. Compare original and changed predictions at matching original vertices. Polygon faces are used for connectivity; the original triangulation is used only to interpolate plots.

In [ ]:
MAX_TRIANGLES_PER_ELEMENT = 3
COLLAPSE_PERCENT = 10.0

experiments, meshes = U.graph_variants(
    sample, POINT_RADIUS, MAX_TRIANGLES_PER_ELEMENT, COLLAPSE_PERCENT, SEED
)
_ = U.plot_connectivities(graphs, experiments, center=(0.03, 0), half_width=0.025)


In [ ]:
changed = {
    label: {name: U.predict_graph(model, variant[name], DEVICE) for name, model in models.items()}
    for label, (variant, _, _) in experiments.items()
}
_ = U.plot_drift(sample, experiments, original_predictions, changed, quantile=0.99)


Try to explain in which scenarios do mesh and poincloud based models drift (i.e., fail to provide the same estimate at the same physical location) and why.

## 2. Neural field property · adressing the drift
A neural field evaluates `fθ(x,y) → (u,v)` independently for each query. No edges or other query points enter its forward pass.

**Exercise 2:** Fit neural field to exactly the previous section's CFD target; compare approximation error and remenshing drift with both GNNs. 

In [ ]:
class NeuralField(nn.Module):
    def __init__(self, levels=6, width=128):
        super().__init__()
        self.coords = FourierCoordinates(2, levels)
        self.net = nn.Sequential(
            nn.Linear(self.coords.out_dim, width),
            nn.SiLU(),
            nn.Linear(width, width),
            nn.SiLU(),
            nn.Linear(width, 2),
        )

    def forward(self, pos):
        # TODO 2: encode coordinates and apply the network.
        raise NotImplementedError("Complete exercise 2")


FIELD_STEPS = 2000
U.set_seed(SEED + 2)

field = NeuralField().to(DEVICE)


def field_loss(model):
    # TODO 2: supervised snapshot loss.
    raise NotImplementedError("Complete exercise 2")


history = U.train_loop(field, field_loss, FIELD_STEPS, lr=1e-3)
U.plot_history({"Neural field": history})

field_full = U.query_field(field, sample.x, DEVICE)
_ = U.plot_predictions(sample, {"Neural field": field_full})


In [ ]:
field_changed = {}

for label, (variant, ids, face) in experiments.items():
    values = U.query_field(field, variant["mesh"].x, DEVICE)
    field_changed[label] = {"Neural field": values}

combined_original = {**original_predictions, "Neural field": field_full}
combined_changed = {label: {**changed[label], **field_changed[label]} for label in experiments}
_ = U.plot_drift(sample, experiments, combined_original, combined_changed, quantile=0.99)


## 3. (Autoregressive) Neural operator · temporal rollouts
The fixed field above has no state input and cannot advance time. Here we exapnd a neural field formulation to a neural operator that can be trained to predict the next velocity field given the current one. The operator is a small neural network that takes as input the current velocity field and its coordinates, and outputs the next velocity field at each query coordinate.

Use a contiguous sequence from this same case, with chronological train/validation/test blocks. This tests temporal continuation, not generalization across geometries.

In [ ]:
SEQUENCE_START = CYLINDER_START - 1
FRAME_STRIDE = 2  # Predict two saved frames ahead; keep fixed across all trials.
TRAIN_TRANSITIONS = 60
VAL_TRANSITIONS = 20
ROLLOUT_STEPS = 60  # Longer than the previous 20-step comparison.
VAL_START = TRAIN_TRANSITIONS + 1
TEST_START = VAL_START + VAL_TRANSITIONS + 1
N_FRAMES = TEST_START + ROLLOUT_STEPS + 1

query_pos, states = U.load_sequence(
    dataset, SEQUENCE_START, N_FRAMES, len(dataset.pos), SEED, stride=FRAME_STRIDE
)

train_ids = torch.arange(TRAIN_TRANSITIONS)
val_ids = torch.arange(VAL_START, VAL_START + VAL_TRANSITIONS)
query_pos = query_pos.to(DEVICE)
states = states.to(DEVICE)


### Exercise 3a · Learn the temporal increment

The network is trained to predict the **increment** between the current and next velocity field ($u_{next} = u_{current} + Δu_θ$) rather than the next field directly. This is a common technique in time series prediction, as it can help the model focus on learning the dynamics of change rather than the absolute values. Each training sample consists of the current velocity field and its coordinates as input.

Train the operator and evaluate its rollout performance on the validation set.

In [ ]:
class AutoregressiveNeuralField(nn.Module):
    def __init__(self, width=128, levels=4):
        super().__init__()
        self.coords = FourierCoordinates(2, levels)
        self.net = nn.Sequential(
            nn.Linear(self.coords.out_dim + 2, width),
            nn.SiLU(),
            nn.Linear(width, width),
            nn.SiLU(),
            nn.Linear(width, width),
            nn.SiLU(),
            nn.Linear(width, width),
        )
        self.head = nn.Sequential(nn.Linear(width, width), nn.SiLU(), nn.Linear(width, 2))

        # Residual head init
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, pos, state):
        # TODO 3a: residual next-state prediction.
        raise NotImplementedError("Complete exercise 3a")


In [ ]:
ROLLOUT_TRAIN_STEPS = 5000


def step_loss(model):
    """Samples different snapshots during training"""
    t = train_ids[torch.randint(len(train_ids), (1,))][0].to(DEVICE)
    # TODO 3a: next-frame loss.
    raise NotImplementedError("Complete exercise 3a")


@torch.no_grad()
def step_validation(model):
    """Validates all unseen snapshots"""
    return (
        torch.stack(
            [
                F.mse_loss(model(query_pos, states[t : t + 1][0]), states[t + 1 : t + 2][0])
                for t in val_ids.tolist()
            ]
        )
        .mean()
        .item()
    )


U.set_seed(SEED + 3)
model = AutoregressiveNeuralField().to(DEVICE)
history = U.train_loop(model, step_loss, ROLLOUT_TRAIN_STEPS, val_fn=step_validation, lr=1e-3)

_ = U.plot_history({"Neural operator": history})


Evaluate two rollout strategies: **Reseeded** and **Pure**:
* **Reseeded:** predict only the next state, at every step use the preceding CFD frame instead of the network's own prediction.
* **Pure:** predict the next state and feed it back as input for the subsequent prediction.

In addition a **Frozen** baseline is provided, which uses the initial state as input for all predictions.

Explain why error can accumulate without reseeding, why it need not increase monotonically, and why query invariance does not imply stable dynamics. Can the reseeding strategy be used in practice?

In [ ]:
@torch.no_grad()
def compare_rollouts(model, pos, current, reference_inputs):
    model.eval()
    pure, reseeded = [], []

    for truth_input in reference_inputs:
        # TODO 3b: reseeded prediction and pure feedback update.
        raise NotImplementedError("Complete exercise 3b")

    return torch.stack(reseeded), torch.stack(pure)


reseeded, pure = compare_rollouts(
    model, query_pos, states[TEST_START], states[TEST_START : TEST_START + ROLLOUT_STEPS]
)

reference = states[TEST_START + 1 : TEST_START + ROLLOUT_STEPS + 1]
sampled_target_frames = SEQUENCE_START + FRAME_STRIDE * torch.arange(
    TEST_START + 1, TEST_START + ROLLOUT_STEPS + 1
)

curves = U.plot_rollout_errors(
    reference,
    reseeded,
    pure,
    states[TEST_START],
    frame_ids=sampled_target_frames,
    title="Full mesh",
)


In [ ]:
# Reuse exactly the target arrays plotted above; all vertices are retained.
rollout_movie = U.animate_rollouts(
    dataset.pos,
    reference,
    reseeded,
    pure,
    frame_ids=sampled_target_frames.numpy(),
    face=dataset.face,
    fps=12,
    component="speed",
    every=1,
    show_errors=True,
    expected_curves=curves,
)


## 4. Neural operators through cross-attention

For now we have operated with a simple neural operator design where each query is processed independently and there is no interaction between queries. This is a strong assumption that can be relaxed and still allow to keep the model query-independent. Let's consider a more general neural operator design that allows for certain interaction between queries utilising cross-attention. The idea is to allow each query to attend to a set of fixed key/value tokens. This allows the model to capture more complex dependencies between queries while still maintaining query independence if key/value tokens stay fixed.

**a) Self-attention** lets every token attend to all tokens in the same set. Given token features $X$: queries, keys, and values are constructed using learned projections $q, k, v$:

$\operatorname{SelfAttention}(X) = \operatorname{softmax}\!\left(\frac{q(X)k(X)^\top}{\sqrt{d_k}}\right)v(X)$

**b) Cross-attention** lets every query token attend to a separate set of key/value tokens. Given query features $X$ and conditioning features $Z$, queries are constructed from $X$ and keys/values from $Z$: (PerceiverIO)

$\operatorname{CrossAttention}(X, Z) = \operatorname{softmax}\!\left(\frac{q(X)k(Z)^\top}{\sqrt{d_k}}\right)v(Z)$

Each query independently reads information from the same conditioning set. If $Z$ stays fixed, adding or removing other queries does not affect the output for an existing query. This enables a conditional neural field formulations. The queries share conditioning information, but do not directly interact with one another.

**c) Anchor-attention** is a special case of cross-attention introduced in [Alkin et al, AB-UPT] where the key/value tokens are a fixed set of so-called *anchor* tokens. Given query features $X$, a sparse subset of *anchor* tokens $A \subset X$ is sampled uniformly from $X$:

$\operatorname{AnchorAttention}(X, A) = \operatorname{softmax}\!\left(\frac{q(X)k(A)^\top}{\sqrt{d_k}}\right)v(A)$

This formulation allows for neural field property by fixing the anchor tokens, while also drastically reducing the computational cost of attention from $O(N^2)$ to $O(NM)$ where $M$ is the number of anchors and $N$ is the number of query tokens. The anchor tokens can be seen as a sparse saptial support set of the input tokens, allowing the model to capture global dependencies while still maintaining query independence.

### Exercise 4a · Implement cross-attention

Project and split Q/K/V, then apply scaled dot-product attention.


In [ ]:
class CrossAttention(nn.Module):
    def __init__(self, width=192, heads=3):
        super().__init__()
        if width % heads:
            raise ValueError("width must be divisible by heads")

        self.heads, self.head_dim = heads, width // heads
        self.q, self.k, self.v, self.out = [nn.Linear(width, width) for _ in range(4)]

    def forward(self, queries, context):
        b, nq, width = queries.shape
        nk = context.shape[1]

        def split(x, n):
            return x.reshape(b, n, self.heads, self.head_dim).transpose(1, 2)

        # TODO 4a: project/split Q,K,V and compute attention.
        raise NotImplementedError("Complete exercise 4a")
        return self.out(answer.transpose(1, 2).reshape(b, nq, width))


### Exercise 4b · Implement the residual anchor operator

Keep the same anchors at both stages. LayerNorm and MLPs operate token-wise.


In [ ]:
class AnchorAttentionNeuralOperator(nn.Module):
    def __init__(self, width: int = 192, heads: int = 3):
        super().__init__()
        self.coords = FourierCoordinates(2, levels=4)
        self.project = nn.Sequential(
            nn.Linear(self.coords.out_dim + 2, width),
            nn.SiLU(),
            nn.Linear(width, width),
        )
        self.attn1 = CrossAttention(width, heads)
        self.attn2 = CrossAttention(width, heads)
        self.mlp1 = nn.Sequential(nn.Linear(width, width), nn.SiLU(), nn.Linear(width, width))
        self.mlp2 = nn.Sequential(nn.Linear(width, width), nn.SiLU(), nn.Linear(width, width))

        self.norm = nn.LayerNorm(width)
        self.head = nn.Sequential(nn.Linear(width, width), nn.SiLU(), nn.Linear(width, 2))

        # Residual head init
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def forward(
        self, pos: torch.Tensor, state: torch.Tensor, anchors_idx: torch.Tensor
    ) -> torch.Tensor:
        unbatched = state.ndim == 2

        if unbatched:
            state = state.unsqueeze(0)

        spatial = self.coords(pos).unsqueeze(0).expand(state.shape[0], -1, -1)

        x = self.project(torch.cat([spatial, state], dim=-1))

        # TODO 4b: two residual attention/MLP stages and residual velocity head.
        raise NotImplementedError("Complete exercise 4b")

        return prediction[0] if unbatched else prediction


### Exercise 4c · Train with anchors only

Sample a uniform subset of vertices on each update. Pass only its coordinates/current velocities to the model, use local anchor indices, and supervise only its next-frame velocities. All training tokens are anchors; extra queries are introduced at inference. Fixed anchor-only validation chooses the checkpoint.


In [ ]:
ANCHORS = min(1024, len(query_pos) // 2)
ATTENTION_STEPS = 10000
assert ANCHORS > 0
# A fixed validation anchor set makes checkpoint selection comparable.
validation_anchor_ids = torch.randperm(len(query_pos), device=DEVICE)[:ANCHORS]


In [ ]:
anchor_ids = torch.randperm(len(query_pos))[:ANCHORS].to(DEVICE)
_ = U.plot_anchor_locations(dataset, query_pos.cpu(), query_pos[anchor_ids].cpu(), radius=None)


In [ ]:
def anchor_attention_loss(model):
    """Uniform subset; only anchors enter the forward pass and loss."""
    # TODO 4c: uniform anchor sampling, compact inputs, local indices and loss.
    raise NotImplementedError("Complete exercise 4c")


@torch.no_grad()
def anchor_attention_validation(model):
    ids = validation_anchor_ids
    local_ids = torch.arange(len(ids), device=DEVICE)
    return (
        torch.stack(
            [
                F.mse_loss(model(query_pos[ids], states[t, ids], local_ids), states[t + 1, ids])
                for t in val_ids.tolist()
            ]
        )
        .mean()
        .item()
    )


U.set_seed(SEED + 4)
attn_field = AnchorAttentionNeuralOperator().to(DEVICE)
attn_history = U.train_loop(
    attn_field,
    anchor_attention_loss,
    ATTENTION_STEPS,
    val_fn=anchor_attention_validation,
    lr=1e-5,
)
attn_field.eval()
U.plot_history({"Anchor-attention neural operator": attn_history})


In [ ]:
@torch.no_grad()
def compare_rollouts(model, pos, current, reference_inputs):
    model.eval()

    anchors_idx = torch.randperm(len(pos), device=DEVICE)[:ANCHORS]
    pure, reseeded = [], []

    for truth_input in reference_inputs:
        reseeded.append(model(pos, truth_input, anchors_idx).cpu())
        current = model(pos, current, anchors_idx)
        pure.append(current.cpu())

    return torch.stack(reseeded), torch.stack(pure)


reseeded, pure = compare_rollouts(
    attn_field, query_pos, states[TEST_START], states[TEST_START : TEST_START + ROLLOUT_STEPS]
)
reference = states[TEST_START + 1 : TEST_START + ROLLOUT_STEPS + 1]
sampled_target_frames = SEQUENCE_START + FRAME_STRIDE * torch.arange(
    TEST_START + 1, TEST_START + ROLLOUT_STEPS + 1
)

rollout_movie = U.animate_rollouts(
    dataset.pos,
    reference,
    reseeded,
    pure,
    frame_ids=sampled_target_frames.numpy(),
    face=dataset.face,
    fps=12,
    component="speed",
    every=1,
    show_errors=True,
)


## 5. Attention, conditioning and the neural field property

### Exercise 5a · Decode together, in chunks, and individually

Freeze the checkpoint, current CFD frame and anchor inputs. Each query reads the same evolving anchor representations, and anchors never read non-anchor queries. Therefore splitting, removing or reordering queries preserves predictions at retained coordinates. LayerNorm and the MLPs act independently on each token.

Evaluate the full mesh together and in chunks. Also test individual decoding on 32 non-anchor points; running every vertex individually is unnecessarily slow. Floating-point differences are allowed, so inspect numerical drift and use a tolerance. A small drift is different from a small CFD error.


In [ ]:
EVALUATION_STEP = TEST_START
QUERY_CHUNK = 256
ATOL = 3e-3 if DEVICE.type == "cuda" else 3e-5
RTOL = 3e-3 if DEVICE.type == "cuda" else 3e-4
U.set_seed(SEED + 5)
fixed_anchors = torch.randperm(len(query_pos), device=DEVICE)[:ANCHORS]
current = states[EVALUATION_STEP]
truth = states[EVALUATION_STEP + 1].cpu()
view = Data(pos=dataset.pos, face=dataset.face, y=truth)
attn_field.eval()

joint = U.predict_anchor_snapshot(attn_field, query_pos, current, fixed_anchors)[0]
# TODO 5a: decode in separate chunks with exactly the same anchors.
raise NotImplementedError("Complete exercise 5a")
mask = torch.ones(len(query_pos), dtype=torch.bool, device=DEVICE)
mask[fixed_anchors] = False
probe_ids = torch.arange(len(query_pos), device=DEVICE)[mask][:32]
individual = U.query_anchor_field(
    attn_field,
    query_pos[fixed_anchors],
    current[fixed_anchors],
    query_pos[probe_ids],
    current[probe_ids],
    chunk_size=1,
)[0]
# TODO 5a: compare retained predictions with numerical tolerance.
raise NotImplementedError("Complete exercise 5a")
assert torch.allclose(joint[probe_ids], individual, atol=ATOL, rtol=RTOL)
print("Chunked relative drift:", U.rel_l2(separate, joint))
print("Individual-query relative drift:", U.rel_l2(individual, joint[probe_ids]))
_ = U.plot_attention_maps(
    view,
    {"Together": joint, "Separate chunks": separate},
    baseline=joint,
    title="Fixed anchors · same input frame",
)


### Exercise 5b · Change the anchors

Query independence is conditional on the anchor support and its current-state values. Replace half the anchors, then sample a new set of the same size. Keep the model, original vertices, query inputs and target frame unchanged. Plot CFD error and prediction drift relative to the original anchor set.

In [ ]:
new_anchors = torch.randperm(len(query_pos), device=DEVICE)[:ANCHORS]
anchor_sets = {
    "Original anchors": fixed_anchors,
    "New anchors": new_anchors,
}
anchor_predictions = {}

for label, ids in anchor_sets.items():
    _ = U.plot_anchor_locations(dataset, dataset.pos, dataset.pos[ids.cpu()])
    anchor_predictions[label] = U.predict_anchor_snapshot(
        attn_field,
        query_pos,
        current,
        ids,
        chunk_size=QUERY_CHUNK,
    )[0]
    
_ = U.plot_attention_maps(
    view,
    anchor_predictions,
    baseline=joint,
    title="Changed anchor conditioning",
)


### Exercise 5c · Anchor-attention vs self-attention

Compare computational resources of anchor-attention and self-attention operator - if training self-attention operator takes too long, train anchor-attention operators with increasing number of anchors to extrapolate the trend.

In [ ]:
class SelfAttentionNeuralOperator(AnchorAttentionNeuralOperator):
    def forward(self, pos, state, anchors_idx=None):
        all_tokens = torch.arange(len(pos), device=pos.device)
        return super().forward(pos, state, all_tokens)


COST_EPOCHS = 10  
COST_WIDTH = 64
COST_HEADS = 2
COST_LR = 1e-4

cost_models, epoch_records = {}, []

def cost_loss(net, t, mode, anchors):
    if mode.startswith("Anchor"):
        ids = torch.randperm(len(query_pos), device=DEVICE)[:anchors]
        return F.mse_loss(
            net(query_pos[ids], states[t, ids], torch.arange(len(ids), device=DEVICE)),
            states[t + 1, ids],
        )
    return F.mse_loss(net(query_pos, states[t]), states[t + 1])


for label, anchors, model_type in [
    ("Anchor-attention-1024", 1024, AnchorAttentionNeuralOperator),
    ("Anchor-attention-2048", 2048, AnchorAttentionNeuralOperator),
    ("Anchor-attention-4096", 4096, AnchorAttentionNeuralOperator),
    # ("Self-attention", -1, SelfAttentionNeuralOperator),
]:
    U.set_seed(SEED + 7)
    net = model_type(COST_WIDTH, COST_HEADS).to(DEVICE)
    records = U.train_profiled_epochs(
        net,
        lambda model, t: cost_loss(model, t, label, anchors),
        train_ids.tolist(),
        epochs=COST_EPOCHS,
        lr=COST_LR,
        label=label,
    )
    epoch_records.extend(records)
    if records[-1]["status"] == "ok":
        cost_models[label] = net.cpu().eval()
    del net

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

_ = U.plot_epoch_costs(epoch_records)
U.display_records([r for r in epoch_records if r["epoch"] == COST_EPOCHS or r["status"] != "ok"])
